# WasteWise AI: cloud training (Kaggle / Colab)

Trains the YOLOv8s baseline on the prepared 6-class dataset, evaluates it on the held-out
**test** split, and packs everything into `wastewise_results.zip`.

**Create the upload locally first:** `python ml/package_for_cloud.py` creates `dist/wastewise_cloud.zip`.

### Kaggle (recommended: runs headless for up to 12 h)
1. *Create → New Dataset*: upload `wastewise_cloud.zip`. Kaggle extracts it automatically.
2. *Create → New Notebook → File → Import Notebook*: this file. Then *Add Input*: your dataset.
3. *Settings*: **Accelerator = GPU T4 x2 or P100**, **Internet = On** (a phone-verified account is required).
4. *Save Version → Save & Run All (Commit)*. You can close the browser; it keeps running.
5. When it finishes, open the version → *Output* → download **`wastewise_results.zip`**.

### Colab
1. Upload `wastewise_cloud.zip` to the root of your Google Drive (`MyDrive/`).
2. *Runtime → Change runtime type → T4 GPU*, then *Run all*. Keep the tab open, because free Colab
   disconnects idle sessions. The results zip is copied back to `MyDrive/`.

### Back on your PC
Extract `wastewise_results.zip` **into the project root**. It contains `models/best.pt`,
`models/best.json`, `runs/detect/yolov8s_baseline/` and `runs/evaluate/...` at the right paths.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
PROJECT = WORK / "wastewise"


def find_source() -> Path:
    if ON_KAGGLE:
        for p in Path("/kaggle/input").rglob("train.py"):
            if p.parent.name == "ml" and p.parent.parent.name == "wastewise":
                return p.parent.parent          # Kaggle auto-extracted the zip
        for p in Path("/kaggle/input").rglob("wastewise_cloud.zip"):
            return p
    if ON_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        candidate = Path("/content/drive/MyDrive/wastewise_cloud.zip")
        if candidate.exists():
            return candidate
    raise FileNotFoundError("wastewise_cloud.zip (or an extracted wastewise/ folder) not found")


if not (PROJECT / "ml" / "train.py").exists():
    src = find_source()
    print("Source:", src)
    if src.is_dir():
        shutil.copytree(src, PROJECT)      # inputs are read-only; Ultralytics writes label caches
    else:
        with zipfile.ZipFile(src) as zf:
            zf.extractall(WORK)
os.chdir(PROJECT)
WORKERS = min(4, os.cpu_count() or 2)
print("Project:", Path.cwd(), "| dataloader workers:", WORKERS)

In [ ]:
# Same Ultralytics version as the local environment, for reproducibility.
!pip install -q "ultralytics==8.4.165"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python ml/check_env.py --skip-inference

In [ ]:
!python ml/verify_dataset.py

In [ ]:
# Train (auto-resumes from last.pt if the process dies). --promote writes models/best.pt + best.json.
RUN = PROJECT / "runs/detect/yolov8s_baseline"
for attempt in range(1, 4):
    if (PROJECT / "models/best.json").exists():
        break
    if (RUN / "weights/last.pt").exists():
        print(f"Attempt {attempt}: resuming interrupted run")
        get_ipython().system("python ml/train.py --name yolov8s_baseline --resume --promote")
    else:
        get_ipython().system(f"python ml/train.py --workers {WORKERS} --promote")

assert (PROJECT / "models/best.json").exists(), "Training did not complete, see output above"
print((PROJECT / "models/best.json").read_text())

In [ ]:
# Held-out test-set evaluation (overall, per class, per source).
!python ml/evaluate.py --workers {WORKERS}

In [ ]:
out = WORK / "wastewise_results.zip"
files = [PROJECT / "models/best.pt", PROJECT / "models/best.json"]
for folder in ("runs/detect/yolov8s_baseline", "runs/evaluate"):
    files += [p for p in (PROJECT / folder).rglob("*") if p.is_file() and p.suffix != ".cache"]

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p, p.relative_to(PROJECT).as_posix())
print(f"{out}  {out.stat().st_size / 1e6:.1f} MB, {len(files)} files")

if ON_COLAB:
    shutil.copy(out, "/content/drive/MyDrive/wastewise_results.zip")
    print("Copied to MyDrive/wastewise_results.zip")